# 🤖 Prediction Model Development

Train, evaluate, and iterate on ML models to predict heat pump energy consumption
from weather forecasts. This notebook provides a deeper, more interactive environment
compared to the production `app/services/prediction.py`.

**Goal**: Build the best model to predict hourly electrical + thermal energy from weather features.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

from app.database.database import get_db_session
from app.database.models import Sensor, SensorReading, Location
from app.services.weather import fetch_historical

print("✅ Imports ready")

✅ Imports ready


## 1. Load Energy & Weather Data

Load heat pump electrical energy (cumulative meter) and compute hourly consumption deltas. Then fetch historical weather from Open-Meteo for Bonn, Germany.

In [ ]:
with get_db_session() as db:
    # Load electrical energy meter (cumulative kWh)
    elec_sensor = db.query(Sensor).filter(Sensor.name == 'warmepumpe_Energie_sum').first()
    rows = (db.query(SensorReading.timestamp, SensorReading.value)
            .filter(SensorReading.sensor_id == elec_sensor.id)
            .order_by(SensorReading.timestamp).all())

energy = pd.DataFrame(rows, columns=['timestamp', 'cumulative_kwh'])
energy['timestamp'] = pd.to_datetime(energy['timestamp'], utc=True)
energy = energy.set_index('timestamp').resample('1h').last().dropna()
energy['consumption_kwh'] = energy['cumulative_kwh'].diff().clip(lower=0)
energy = energy.dropna()

# Fetch weather history from Open-Meteo (sync)
import httpx

lat, lon = 50.7374, 7.0982  # Bonn
params = {
    'latitude': lat, 'longitude': lon,
    'start_date': energy.index.min().strftime('%Y-%m-%d'),
    'end_date': energy.index.max().strftime('%Y-%m-%d'),
    'hourly': 'temperature_2m,relative_humidity_2m,wind_speed_10m,cloud_cover,direct_radiation',
    'timezone': 'Europe/Berlin'
}
with httpx.Client(timeout=60) as client:
    r = client.get('https://archive-api.open-meteo.com/v1/archive', params=params)
    weather_json = r.json()

hourly = weather_json['hourly']
weather = pd.DataFrame({
    'timestamp': pd.to_datetime(hourly['time'], utc=True),
    'temperature': hourly['temperature_2m'],
    'humidity': hourly['relative_humidity_2m'],
    'wind_speed': hourly['wind_speed_10m'],
    'cloud_cover': hourly['cloud_cover'],
    'solar_radiation': hourly['direct_radiation'],
})
weather = weather.set_index('timestamp')

# Merge
df = energy[['consumption_kwh']].join(weather, how='inner').dropna()
print(f"✅ Dataset: {len(df)} hourly samples, {df.index.min().date()} → {df.index.max().date()}")
df.head()

2026-03-24 15:26:31,967 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-24 15:26:31,968 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:26:32,025 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-24 15:26:32,025 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:26:32,049 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-24 15:26:32,049 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:26:32,118 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:26:32,122 INFO sqlalchemy.engine.Engine SELECT api_sensors.id AS api_sensors_id, api_sensors.device_id AS api_sensors_device_id, api_sensors.name AS api_sensors_name, api_sensors.description AS api_sensors_description, api_sensors.sensor_type_id AS api_sensors_sensor_type_id, api_sensors.location_id AS api_sensors_location_id, api_sensors.manufacturer AS api_sensors_manufacturer, api_sensors.model AS api_sensors_model, api_sensors.firmware_version AS api_sensors_

RuntimeError: This event loop is already running

## 2. Feature Engineering

Create time-based cyclical features (hour-of-day, day-of-week using sin/cos encoding), polynomial temperature terms, and interaction features.

In [ ]:
# Time-based cyclical features
df['hour'] = df.index.hour
df['dow'] = df.index.dayofweek
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)

# Polynomial & interaction features
df['temperature_sq'] = df['temperature'] ** 2
df['temp_x_wind'] = df['temperature'] * df['wind_speed']
df['temp_x_humidity'] = df['temperature'] * df['humidity']

# Lag features (previous hours)
for lag in [1, 2, 3]:
    df[f'consumption_lag_{lag}'] = df['consumption_kwh'].shift(lag)
df = df.dropna()

feature_cols = [
    'temperature', 'humidity', 'wind_speed', 'cloud_cover', 'solar_radiation',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'temperature_sq', 'temp_x_wind', 'temp_x_humidity',
    'consumption_lag_1', 'consumption_lag_2', 'consumption_lag_3',
]

X = df[feature_cols]
y = df['consumption_kwh']
print(f"✅ Features: {len(feature_cols)} columns, {len(X)} samples")
X.describe().round(2)

## 3. Model Comparison with Time-Series Cross-Validation

Compare three models using `TimeSeriesSplit` to respect temporal ordering. We evaluate MAE and R² across 5 folds.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42),
}

results = []
for name, model in models.items():
    mae_scores = -cross_val_score(model, X_scaled, y, cv=tscv, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(model, X_scaled, y, cv=tscv, scoring='r2')
    results.append({
        'Model': name,
        'MAE (mean)': mae_scores.mean(),
        'MAE (std)': mae_scores.std(),
        'R² (mean)': r2_scores.mean(),
        'R² (std)': r2_scores.std(),
    })
    print(f"  {name}: MAE={mae_scores.mean():.4f}±{mae_scores.std():.4f}, R²={r2_scores.mean():.4f}±{r2_scores.std():.4f}")

results_df = pd.DataFrame(results)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Mean Absolute Error ↓', 'R² Score ↑'])
fig.add_trace(go.Bar(x=results_df['Model'], y=results_df['MAE (mean)'],
                     error_y=dict(type='data', array=results_df['MAE (std)']),
                     marker_color=['#636EFA', '#EF553B', '#00CC96']), row=1, col=1)
fig.add_trace(go.Bar(x=results_df['Model'], y=results_df['R² (mean)'],
                     error_y=dict(type='data', array=results_df['R² (std)']),
                     marker_color=['#636EFA', '#EF553B', '#00CC96']), row=1, col=2)
fig.update_layout(title='Model Comparison (5-Fold Time-Series CV)', showlegend=False, height=400)
fig.show()
results_df.round(4)

## 4. Feature Importance Analysis

Train the best model (Gradient Boosting) on full data and inspect which features drive predictions.

In [ ]:
# Train Gradient Boosting on full dataset
best_model = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42)
best_model.fit(X_scaled, y)

importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig = go.Figure(go.Bar(
    x=importances.values, y=importances.index,
    orientation='h',
    marker_color=px.colors.sequential.Viridis_r[:len(importances)],
    text=importances.values.round(3), textposition='outside'
))
fig.update_layout(
    title='Feature Importance — Gradient Boosting',
    xaxis_title='Importance', yaxis_title='',
    height=500, margin=dict(l=160)
)
fig.show()

## 5. Residual Analysis

Check prediction errors for systematic bias. A good model should have residuals that are normally distributed and centered around zero.

In [ ]:
y_pred = best_model.predict(X_scaled)
residuals = y - y_pred

fig = make_subplots(rows=1, cols=2, subplot_titles=['Predicted vs Actual', 'Residual Distribution'])

# Scatter: predicted vs actual
fig.add_trace(go.Scattergl(
    x=y, y=y_pred, mode='markers', marker=dict(size=2, opacity=0.3, color='#636EFA'),
    name='Samples'
), row=1, col=1)
max_val = max(y.max(), y_pred.max())
fig.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode='lines',
                         line=dict(dash='dash', color='red'), name='Perfect'), row=1, col=1)
fig.update_xaxes(title_text='Actual (kWh)', row=1, col=1)
fig.update_yaxes(title_text='Predicted (kWh)', row=1, col=1)

# Histogram: residuals
fig.add_trace(go.Histogram(x=residuals, nbinsx=80, marker_color='#EF553B', name='Residuals'), row=1, col=2)
fig.update_xaxes(title_text='Residual (kWh)', row=1, col=2)

mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)
fig.update_layout(
    title=f'Residual Analysis — MAE: {mae:.4f} kWh, R²: {r2:.4f}',
    height=450, showlegend=False
)
fig.show()

print(f"Residual stats: mean={residuals.mean():.5f}, std={residuals.std():.4f}, "
      f"skew={residuals.skew():.3f}, kurtosis={residuals.kurtosis():.3f}")

## 6. Prediction vs Actual — Time Series View

Plot a two-week window comparing predictions against measured consumption to visually assess tracking quality.

In [ ]:
# Pick a representative 2-week winter window
mid = len(df) // 2
window = slice(mid, mid + 24*14)
ts_window = df.index[window]

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_window, y=y.iloc[window], mode='lines', name='Actual',
                         line=dict(color='#636EFA')))
fig.add_trace(go.Scatter(x=ts_window, y=y_pred[window], mode='lines', name='Predicted',
                         line=dict(color='#EF553B', dash='dot')))
fig.update_layout(
    title='14-Day Prediction vs Actual — Heat Pump Consumption',
    xaxis_title='Time', yaxis_title='Consumption (kWh)',
    height=400, legend=dict(x=0.01, y=0.99)
)
fig.show()

## 7. Error by Temperature Band & Hour

Investigate when the model struggles most — low temperatures or specific hours — to guide future improvements.

In [ ]:
analysis = df[['temperature', 'hour', 'consumption_kwh']].copy()
analysis['predicted'] = y_pred
analysis['abs_error'] = np.abs(analysis['consumption_kwh'] - analysis['predicted'])
analysis['temp_band'] = pd.cut(analysis['temperature'], bins=[-20, -5, 0, 5, 10, 15, 20, 35],
                               labels=['<-5', '-5–0', '0–5', '5–10', '10–15', '15–20', '>20'])

fig = make_subplots(rows=1, cols=2, subplot_titles=['MAE by Temperature Band', 'MAE by Hour of Day'])

# By temperature band
temp_err = analysis.groupby('temp_band', observed=True)['abs_error'].mean()
fig.add_trace(go.Bar(x=temp_err.index.astype(str), y=temp_err.values,
                     marker_color='#AB63FA'), row=1, col=1)
fig.update_xaxes(title_text='Temperature (°C)', row=1, col=1)
fig.update_yaxes(title_text='MAE (kWh)', row=1, col=1)

# By hour
hour_err = analysis.groupby('hour')['abs_error'].mean()
fig.add_trace(go.Bar(x=hour_err.index, y=hour_err.values,
                     marker_color='#FFA15A'), row=1, col=2)
fig.update_xaxes(title_text='Hour of Day', row=1, col=2)
fig.update_yaxes(title_text='MAE (kWh)', row=1, col=2)

fig.update_layout(title='Error Analysis by Conditions', height=400, showlegend=False)
fig.show()

## 8. Model Export

Save the trained model and scaler for use by the prediction API service.

In [ ]:
import joblib, json

model_dir = os.path.join('..', 'models')
os.makedirs(model_dir, exist_ok=True)

joblib.dump(best_model, os.path.join(model_dir, 'heatpump_gb_model.joblib'))
joblib.dump(scaler, os.path.join(model_dir, 'heatpump_scaler.joblib'))

meta = {
    'model': 'GradientBoostingRegressor',
    'features': feature_cols,
    'n_samples': len(X),
    'date_range': f"{df.index.min().isoformat()} → {df.index.max().isoformat()}",
    'mae': float(mae),
    'r2': float(r2),
    'trained_at': datetime.now().isoformat(),
}
with open(os.path.join(model_dir, 'model_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f"✅ Model saved to {model_dir}/")
print(f"   MAE: {mae:.4f} kWh | R²: {r2:.4f}")
print(f"   Features: {len(feature_cols)} | Samples: {len(X)}")